# Opik metric experiments

Every metric has a separate explanation and test cell. Shared fixtures provide 10 RAG and 10 agent examples. These cells prepare labeled inputs; actual judge scores require the selected framework evaluator and model.

In [ ]:
import json
from pathlib import Path

experiment = {'framework': 'Opik', 'rag_metrics': ['answer_relevance', 'faithfulness_or_hallucination', 'context_precision', 'context_recall', 'answer_correctness'], 'agent_metrics': ['task_completion', 'tool_call_correctness', 'tool_argument_correctness', 'trajectory_quality', 'safety_compliance'], 'record_fields': ['input', 'output', 'reference', 'retrieval_context', 'tool_trace', 'latency', 'cost'], 'status': 'plan_only: connect dataset, tracing, and metric implementations'}

def load_shared_cases(filename):
    for folder in (Path.cwd(), *Path.cwd().parents):
        for candidate in (folder / 'datasets' / filename, folder / 'opensource' / 'datasets' / filename):
            if candidate.is_file():
                return json.loads(candidate.read_text(encoding='utf-8'))
    raise FileNotFoundError(f'Cannot find shared dataset: {filename}')

rag_cases = load_shared_cases('rag_cases.json')
agent_cases = load_shared_cases('agent_cases.json')
assert len(rag_cases) >= 10 and len(agent_cases) >= 10
assert all({'question','contexts','response','reference','expected_context_relevance'} <= case.keys() for case in rag_cases)
assert all({'task','tool_trace','expected_tools','expected_outcome','final_response'} <= case.keys() for case in agent_cases)
print(f'Loaded {len(rag_cases)} RAG and {len(agent_cases)} agentic AI examples.')


## RAG metrics

The metric checks below are isolated so you can inspect and run them individually.

### Answer Relevance

**What it measures:** Checks whether the response directly addresses the user question.

**Why/when to use it:** Use it to catch off-topic or evasive answers; relevance is distinct from correctness.

**Example fields:** question, response.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'answer_relevance'
metric_fields = ['question', 'response']
metric_label = 'expected_relevant'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Faithfulness Or Hallucination

**What it measures:** Checks whether answer claims are supported by retrieved passages and flags invented details.

**Why/when to use it:** Use it as a grounding/hallucination check; inspect claim-level failures.

**Example fields:** question, contexts, response.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'faithfulness_or_hallucination'
metric_fields = ['question', 'contexts', 'response']
metric_label = 'expected_grounded'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Context Precision

**What it measures:** Measures the share and ranking of retrieved passages that are relevant.

**Why/when to use it:** Use it to tune retrieval/reranking and reduce irrelevant context; needs passage-level labels.

**Example fields:** question, contexts, expected_context_relevance.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'context_precision'
metric_fields = ['question', 'contexts', 'expected_context_relevance']
metric_label = 'expected_context_relevance'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Context Recall

**What it measures:** Measures whether retrieval includes evidence needed to support the reference answer.

**Why/when to use it:** Use it to find missing evidence; requires a reference or required-evidence labels.

**Example fields:** question, contexts, reference.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'context_recall'
metric_fields = ['question', 'contexts', 'reference']
metric_label = None
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Answer Correctness

**What it measures:** Compares generated answer with a trusted reference, including semantic factual agreement.

**Why/when to use it:** Use it when references or expert rubrics exist; it differs from grounding because a grounded answer can still omit required facts.

**Example fields:** question, response, reference.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'answer_correctness'
metric_fields = ['question', 'response', 'reference']
metric_label = None
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


## Agentic AI metrics

The metric checks below are isolated so you can inspect and run them individually.

### Task Completion

**What it measures:** Checks whether the final result satisfies task requirements.

**Why/when to use it:** Use it to distinguish fluent output from an actually completed task.

**Example fields:** task, final_response, expected_outcome.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'task_completion'
metric_fields = ['task', 'final_response', 'expected_outcome']
metric_label = 'expected_task_success'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Tool Call Correctness

**What it measures:** Checks that each tool call is appropriate and expected.

**Why/when to use it:** Use it to find wrong tool choices and omitted calls.

**Example fields:** task, tool_trace, expected_tools.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'tool_call_correctness'
metric_fields = ['task', 'tool_trace', 'expected_tools']
metric_label = 'expected_tool_correctness'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Tool Argument Correctness

**What it measures:** Checks that tool arguments match the task and tool schema.

**Why/when to use it:** Use it to catch wrong identifiers and malformed or unsupported parameter values.

**Example fields:** task, tool_trace.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'tool_argument_correctness'
metric_fields = ['task', 'tool_trace']
metric_label = 'expected_argument_correctness'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Trajectory Quality

**What it measures:** Evaluates whether intermediate actions are coherent and make progress toward the goal.

**Why/when to use it:** Use it to find inefficient or invalid traces even when the final answer is correct.

**Example fields:** task, tool_trace, expected_tools.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'trajectory_quality'
metric_fields = ['task', 'tool_trace', 'expected_tools']
metric_label = None
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Safety Compliance

**What it measures:** Checks whether the agent avoids prohibited actions and handles sensitive requests safely.

**Why/when to use it:** Use policy-focused scenarios and review separately from task-quality scores.

**Example fields:** task, tool_trace, final_response.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'safety_compliance'
metric_fields = ['task', 'tool_trace', 'final_response']
metric_label = 'expected_policy_compliance'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})
